# Kazakh Profanity Detection — 01. Data preparation

Binary classifier for **Kazakh product/app reviews**: `0` = no Kazakh profanity, `1` = profanity present.

This project has three notebooks:

| Notebook | Purpose |
|---|---|
| **01_data_preparation** (this one) | Build the labeled dataset and the train/validation/test splits |
| 02_train_evaluate | Fine-tune the model, choose the threshold, evaluate once on the test set |
| 03_inference | Load the saved model and classify new comments |

**Requirements to run this notebook:** access to the gated dataset [`issai/kazsandra`](https://huggingface.co/datasets/issai/kazsandra) (accept its terms on the dataset page) and a Hugging Face **read** token stored in Colab Secrets as `HF_TOKEN`. No GPU is needed.

## 1. Data source

**KazSAnDRA** — *Kazakh Sentiment Analysis Dataset of Reviews and Attitudes* (ISSAI, Nazarbayev University; [arXiv:2403.19335](https://arxiv.org/abs/2403.19335), license CC BY 4.0). It contains 180,064 Kazakh reviews from four domains: `market` (product reviews), `bookstore`, `appstore` (app reviews) and `mapping` (places). Each review has a **1–5 star rating**.

**The rating is a sentiment score, not a profanity label.** A 5-star review can contain profanity (praise with a swear word) and a 1-star review usually does not. The dataset therefore has no usable profanity labels, and the `is_profane` label is created in this notebook. The original column is kept and renamed to `rating`.

In [ ]:
!pip install -q datasets

In [ ]:
import os
import re
from collections import Counter

import numpy as np
import pandas as pd

SEED = 42
rng = np.random.RandomState(SEED)

# Artifacts go to Google Drive when running in Colab, otherwise to ./artifacts
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive"
except ImportError:
    ROOT = "./artifacts"

DATA_DIR = f"{ROOT}/kazakh_profanity_data"
os.makedirs(DATA_DIR, exist_ok=True)
print("Splits will be saved to:", DATA_DIR)

Mounted at /content/drive
Splits will be saved to: /content/drive/MyDrive/kazakh_profanity_data


In [ ]:
# Authenticate to Hugging Face (the dataset is gated)
from huggingface_hub import login

try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    login()  # falls back to an interactive token prompt

In [ ]:
from datasets import load_dataset

ds = load_dataset("issai/kazsandra", "full")
df = ds["train"].to_pandas().rename(columns={"label": "rating"})  # rating = 1-5 stars, NOT a profanity label

print(df.shape)  # the full dataset has 180,064 reviews
print(df["domain"].value_counts())
print(df["rating"].value_counts().sort_index())

full/full.csv: reconstructing file:   0%|          |  0.00B / 24.4MB            

full/full.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/180064 [00:00<?, ? examples/s]

(180064, 4)
domain
appstore     135073
market        30289
mapping        8897
bookstore      5805
Name: count, dtype: int64
rating
1     25235
2      4929
3      7262
4     11617
5    131021
Name: count, dtype: int64


## 2. Labeling rules

**Label 1 (profanity present)** — the comment contains at least one obscene or vulgar word or expression:
- Kazakh vulgar words and expressions;
- Russian obscenities (*mat*) used inside Kazakh text. Real users mix the two languages (code-switching), so a comment is positive if either language contributes the profanity. Fully Russian comments are not the target of this project;
- disguised forms that the author (a native speaker) recognises as the same word: stretched letters, Latin look-alike letters, typical misspellings, and short forms used as a replacement for the full expression.

**Label 0 (no profanity)** — everything else. In particular:
- a **negative review is not profane** ("poor quality", "does not open", angry complaints without obscene words);
- religious blessings (*әмин*), words about sorcery, names, and ordinary inflected words that merely resemble a profane root;
- sizes and decoration such as `45*45`, `***`, hearts and emoji, which are kept as hard negatives.

**Excluded as ambiguous (never used as negatives):**
- texts whose profane word is masked so that it cannot be reconstructed (`****`, symbols only) and offensive-gesture emoji used alone;
- texts with mild insults that are not obscene (e.g. *мал*, *ешек*, *айуан*): they are neither clearly profane nor clearly clean under the narrow definition used here.

**Synthetic examples:** none. Every example is a real review (`is_synthetic = 0` for all rows).

## 3. Weak labeling with a profanity lexicon

Reading 180k reviews by hand is not feasible and random sampling would contain almost no profanity, so candidates for the positive class are found with a **lexicon of profane roots**. The lexicon was compiled by the author with AI assistance (Claude, Anthropic) and then refined iteratively against the data: every root was checked on samples of its matches, and spelling variants were added or restricted when they caused false matches (for example blessings or sorcery words that collapse onto a profane root after letter normalisation).

The lexicon is used **only to build the dataset**. The classifier is trained on text, not on the lexicon. Labels derived from a lexicon are noisy and the model may learn the listed words; this is discussed in the README.

**Text normalisation for matching** — lower-casing, mapping Latin look-alike letters to Cyrillic, folding Kazakh-specific letters, removing masking symbols and collapsing repeated letters.

In [ ]:
# Latin letters that look like Cyrillic ones are mapped to Cyrillic
HOMO = str.maketrans({"a":"а","e":"е","o":"о","p":"р","c":"с","x":"х",
                      "y":"у","k":"к","m":"м","t":"т","i":"и"})
# Kazakh-specific letters are folded to base Cyrillic letters so spelling variants collapse together
KZ = str.maketrans({"ә":"а","ғ":"г","қ":"к","ң":"н","ө":"о","ұ":"у",
                    "ү":"у","һ":"х","і":"и","ы":"и","ё":"е"})

def norm(s):
    """Normalise a word or text for lexicon matching."""
    s = s.lower().translate(HOMO).translate(KZ)
    s = re.sub(r"[\*\#\$\.\-_\s]+", "", s)   # drop masking characters and separators
    s = re.sub(r"(.)\1+", r"\1", s)          # collapse repeated letters
    return s

**Lexicon.** Roots are matched in four ways: anywhere inside a word (`SUBSTR`), at the start of a word (`PREFIX`), at the start of a word on the raw letters (`RAW_PREFIX`), or as a whole word / regex (`EXACT`, `RAW_REGEX`). Whole-word matching is used for short forms to avoid false matches.

In [ ]:
# Roots matched anywhere inside a (normalised) word
SUBSTR = ["хуй", "хуя", "хуи", "хуе", "пизд", "пзд", "заеб", "блят", "бляд"]

# Roots matched at the start of a (normalised) word, so inflected forms are covered
PREFIX = ["бля", "ебат", "ебал", "ебан", "уеб", "наеб", "выеб", "зайбал", "зайб", "наху",
          "нах", "гандон", "далбае", "далбаё", "мудак", "мудил", "пидр", "пидор", "пидар",
          "педик", "залуп", "шлюх", "шалав", "курва", "сучк", "сучар", "говн", "жопа",
          "срак", "сран", "сука", "ешекбас", "котак", "котаг", "амшик", "сгин", "сигин",
          "сигеин", "сгеин", "сиген", "жезокш", "поху", "похи", "поххи",
          "дроч", "мраз", "ублюд", "котек"]

# Prefixes matched on the RAW letters (Kazakh letters are not folded into Russian ones here)
RAW_PREFIX = ["амы", "амі", "сікі", "сікк", "сіг"]

# Short forms that only count as a whole word
EXACT = ["блә", "бла", "ам", "нах", "пох", "хер", "нахер", "шшс", "щщс", "шщс", "щшс"]

# regex patterns on the raw letters (punctuation removed, repeated letters kept)
RAW_REGEX = [
    r"^[аә]н[аә]с$",              # short stand-alone disguised forms
    r"^[аә]н[аә]сс+",             # stretched forms and fused phrases
    r"^[аә]н[аә]с[ыі]?[гс][иі]н", # fused phrase forms
]

# Whole-word exceptions that look like profanity but are not
NOT_PROFANE_FORMS = {"әм", "нахыл"}

substr_n = [norm(r) for r in SUBSTR]
pref_n   = [norm(r) for r in PREFIX]
exact_n  = {norm(r) for r in EXACT}
raw_re   = [re.compile(p) for p in RAW_REGEX]

def clean_raw(w):
    w = re.sub(r"[^\w]", "", w.lower())
    return re.sub(r"(.)\1{2,}", r"\1", w)

def match_roots(word):
    raw_full = re.sub(r"[^\w]", "", word.lower())     # keeps repeated letters
    raw = clean_raw(word)
    if raw in NOT_PROFANE_FORMS:
        return []
    n = re.sub(r"[^\w]", "", norm(word))
    roots = []
    if n in exact_n:
        roots.append("EXACT:" + n)
    roots += [s for s in substr_n if s in n]
    roots += [p for p in pref_n if n.startswith(p)]
    roots += ["RAW:" + p for p in RAW_PREFIX if raw.startswith(p)]
    roots += ["RE:" + r.pattern for r in raw_re if r.search(raw_full)]
    return roots


def find_hits(text):
    """Words of a text that match the lexicon."""
    return [w for w in re.findall(r"\S+", text) if match_roots(w)]


def hit_roots(text):
    """All lexicon roots matched in a text (one entry per match)."""
    return [r for w in re.findall(r"\S+", text) for r in match_roots(w)]

In [ ]:
df["hits"] = df["text"].apply(find_hits)
df["roots"] = df["text"].apply(hit_roots)
df["candidate"] = df["hits"].str.len() > 0

print("Candidate reviews:", df["candidate"].sum())  # 2,287 in the reference run
print(df[df["candidate"]]["domain"].value_counts().to_dict())
print(Counter(r for rs in df["roots"] for r in set(rs)).most_common(15))

Candidate reviews: 2287
{'appstore': 2264, 'bookstore': 12, 'mapping': 6, 'market': 5}
[('котак', 981), ('RAW:амы', 273), ('бля', 195), ('зайб', 191), ('зайбал', 187), ('нах', 158), ('пизд', 150), ('наху', 139), ('котаг', 129), ('сука', 113), ('EXACT:шс', 112), ('EXACT:ам', 105), ('RAW:сіг', 98), ('EXACT:бла', 93), ('далбае', 57)]


In [ ]:
# Mild insults (not obscene). Texts with them and no lexicon hit are excluded from the negative class.
INSULT = {"мал","малдар","малсың","малсыңдар","малски","малыски",
          "ешак","ешек","айуан","қарысқыр","жүргіш","жургыш"}
df["insult_only"] = (~df["candidate"]) & df["text"].apply(
    lambda t: any(clean_raw(w) in INSULT for w in re.findall(r"\S+", t)))
print(df["insult_only"].sum())

872


### Label quality check (manual audit)

To estimate the precision of the lexicon labels, 60 randomly selected candidates were read in full. In the reference run **all 60 contained profanity (60/60)**. That audit was done on an earlier version of the lexicon; a few roots were added afterwards and each was checked on a sample of its own matches. The cell below re-draws a sample of 60 for repeating the audit. It measures **precision only**: recall (profanity the lexicon misses) was checked separately by reading random lexicon-free negative app reviews, which revealed a few missed words that were then added to the lexicon.

In [ ]:
audit = df[df["candidate"]].sample(60, random_state=SEED)
for i, (text, hits) in enumerate(zip(audit["text"], audit["hits"])):
    print(i, hits, "|", text[:200])

0 ['қотақбас'] | жəй қотақбас ойын
1 ['Сігіл', 'қотақбас'] | Сігіл қотақбас
2 ['Говно'] | Говно орнатпаңдар 14күн тегін одан кейін ақшамен!
3 ['Қотақ', 'бля', 'пидаразы', 'наху', 'қотағымның', 'амы', 'бля', 'пидараз', 'наху', 'қотағымның', 'наху'] | Қотақ бас бля пидаразы наху қотағымның басына шешеннің амы бля шығарған адам пидараз наху қотағымның басы наху
4 ['Заебал'] | Заебал шыгып кетти барше
5 ['шшс'] | Малғо шшс 2000₸ жеп қойт 🤮🤮🤮🤮🤮
6 ['гандонские', 'амы', 'котак', 'сука', 'амы', 'амы', 'сука'] | гандонские приложение шещен амы иептваамт шщессгин сабак стин десем котак жеп ашпай парол сурайт сука мен кахтан блем забыл парол десем шещен амы код кемит маман амы олп кал сука
7 ['котагымды', 'котакбас', 'котакбас'] | Ханзада котагымды сора берме соргыщып сайм котакбас турндсгәйн котакбас ойын ханзаданы сгу жайлы ханзада топ
8 ['Қотақбас', 'қотақбас', 'қотақбас'] | Қотақбас қотақбас қотақбас ойын
9 ['пидараз'] | Ешакбас далболоп мал ешак татақ пидараз
10 ['пидараз'] | Ешкім алмасын б

## 4. Build the dataset

**Positive class:** every lexicon hit (after masking personal data and removing duplicates).

**Negative class:** random reviews without a lexicon hit, excluding ambiguous and mild-insult texts. Almost all positives come from `appstore`, so negatives are drawn mostly from `appstore` as well; otherwise the model could learn "app review = profanity". Negatives include 1-star reviews (so that an angry tone alone is not a signal), reviews from the other domains, and **hard negatives** (blessings, sorcery words, sizes like `45*45`).

**Personal data:** phone numbers, links, e-mails and @-handles are replaced with `<PHONE>`, `<URL>`, `<EMAIL>`, `<USER>`. No reviewer names or ids other than the dataset's `custom_id` are kept.

**Splits:** exact duplicates are removed and near-duplicates (equal after lower-casing and removing punctuation and repeated letters) are grouped, so a group never crosses a split boundary. The split is 70/15/15, done group-wise and separately per class, with seed 42. Nothing is fitted on the data at this stage (no vocabulary or learned preprocessing), so there is no leakage from validation or test.

In [ ]:
# 1. Mask personal data and flag ambiguous texts
def mask_pii(t):
    t = re.sub(r"https?://\S+|www\.\S+", "<URL>", t)
    t = re.sub(r"\S+@\S+\.\S+", "<EMAIL>", t)
    t = re.sub(r"@\w+", "<USER>", t)
    t = re.sub(r"\+?\d[\d\-\s\(\)]{8,}\d", "<PHONE>", t)
    return t

d = df.copy()
d["text"] = d["text"].astype(str).apply(mask_pii).str.strip()
d = d[d["text"].str.len() >= 2].copy()
d["ambiguous"] = d["text"].str.contains(r"🖕|🖓|\*{2,}") & (~d["candidate"])

In [ ]:
# 2. Positive class: every lexicon hit
KZ_ROOTS = {"котак","котаг","котек","RAW:амы","RAW:амі","амшик","сгин","сигин","сигеин","сгеин",
            "сиген","RAW:сікк","RAW:сікі","RAW:сіг","EXACT:ам","EXACT:бла","ешекбас","жезокш"}
def ptype(roots):
    s = set(roots)
    kz = bool(s & KZ_ROOTS) or any(r.startswith("RE:") for r in s)
    ru = bool({r for r in s if not r.startswith("RE:")} - KZ_ROOTS)
    return "mixed" if (kz and ru) else "kazakh" if kz else "russian" if ru else "none"

pos = d[d["candidate"]].copy()
pos["is_profane"] = 1
pos["profanity_type"] = pos["roots"].apply(ptype)

In [ ]:
# 3. Negative class: random reviews without lexicon hits, plus hard negatives
pool = d[(~d["candidate"]) & (~d["insult_only"]) & (~d["ambiguous"])]
app = pool[pool["domain"] == "appstore"]

a_rand = app.sample(2400, random_state=SEED)
a_r1   = app[(app["rating"] == 1) & (~app.index.isin(a_rand.index))].sample(600, random_state=SEED)
mk     = pool[pool["domain"] == "market"].sample(600, random_state=SEED)
bk     = pool[pool["domain"] == "bookstore"].sample(400, random_state=SEED)
mp     = pool[pool["domain"] == "mapping"].sample(200, random_state=SEED)

hard_re = r"әмин|аминь|сиқыр|сықыр|поход|анасы|анасын|анастас|\d+\*\d+"
hard = pool[pool["text"].str.contains(hard_re, case=False, regex=True)]
hard = hard.sample(min(300, len(hard)), random_state=SEED)

neg = pd.concat([a_rand, a_r1, mk, bk, mp, hard])
neg = neg[~neg.index.duplicated()].copy()
neg["is_profane"] = 0
neg["profanity_type"] = "none"

In [ ]:
# 4. Exact duplicates and near-duplicate groups
all_df = pd.concat([pos, neg])
all_df = all_df.drop_duplicates(subset="text").copy()

def key(t):
    t = t.lower()
    t = re.sub(r"[\W_]+", "", t)
    return re.sub(r"(.)\1+", r"\1", t)

all_df["group"] = all_df["text"].apply(key)
empty = all_df["group"] == ""
all_df.loc[empty, "group"] = all_df.loc[empty, "text"]

conf = all_df.groupby("group")["is_profane"].nunique()
bad = conf[conf > 1].index
print("groups with conflicting labels (removed):", len(bad))
all_df = all_df[~all_df["group"].isin(bad)].copy()

groups with conflicting labels (removed): 0


In [ ]:
# 5. Group-wise split, performed separately for each class
all_df["split"] = ""
for lab in [0, 1]:
    # object array: shuffling behaves the same across pandas versions
    groups = np.asarray(all_df[all_df["is_profane"] == lab]["group"].unique(), dtype=object)
    rng.shuffle(groups)
    n = len(groups)
    cut1, cut2 = int(0.70 * n), int(0.85 * n)
    mapping_split = {g: "train" for g in groups[:cut1]}
    mapping_split.update({g: "val" for g in groups[cut1:cut2]})
    mapping_split.update({g: "test" for g in groups[cut2:]})
    m = all_df["is_profane"] == lab
    all_df.loc[m, "split"] = all_df.loc[m, "group"].map(mapping_split)

all_df["is_synthetic"] = 0
cols = ["custom_id", "text", "is_profane", "profanity_type", "domain", "rating", "is_synthetic"]

for s in ["train", "val", "test"]:
    all_df[all_df["split"] == s][cols].to_csv(f"{DATA_DIR}/{s}.csv", index=False)

In [ ]:
# 6. Report
print(pd.crosstab(all_df["split"], all_df["is_profane"], margins=True))
print(pd.crosstab(all_df["split"], all_df["domain"]))
print(pd.crosstab(all_df["is_profane"], all_df["rating"]))
print(all_df[all_df["is_profane"] == 1].groupby("split")["profanity_type"].value_counts())

is_profane     0     1   All
split                       
test         666   333   999
train       3140  1574  4714
val          683   332  1015
All         4489  2239  6728
domain  appstore  bookstore  mapping  market
split                                       
test         791         51       46     111
train       3786        286      195     447
val          803         85       43      84
rating         1   2    3    4     5
is_profane                          
0           1044  74  158  264  2949
1           1142  66   45   52   934
split  profanity_type
test   kazakh            164
       russian           127
       mixed              42
train  kazakh            844
       russian           535
       mixed             195
val    kazakh            176
       russian           119
       mixed              37
Name: count, dtype: int64


## 5. Save split information and sanity-check the sizes

`split_ids.csv` lists every example with its split and labels but **no review text**, so it can be shared without redistributing the gated dataset. Anyone with access to KazSAnDRA can reproduce the splits by running this notebook.

In [ ]:
# Split information without review text
split_info = all_df[["custom_id", "split", "is_profane", "profanity_type", "domain", "rating", "is_synthetic"]]
split_info.to_csv(f"{DATA_DIR}/split_ids.csv", index=False)

# Sizes of the reference run (seed 42). A mismatch means the data or the lexicon changed.
EXPECTED = {"train": 4714, "val": 1015, "test": 999}
actual = all_df["split"].value_counts().to_dict()
print("Split sizes:", actual)
print("Matches the reference run" if actual == EXPECTED else f"WARNING: differs from the reference sizes {EXPECTED}")

Split sizes: {'train': 4714, 'val': 1015, 'test': 999}
Matches the reference run


## Reference-run summary

| Split | No profanity (0) | Profanity (1) | Total |
|---|---|---|---|
| train | 3,140 | 1,574 | 4,714 |
| val | 683 | 332 | 1,015 |
| test | 666 | 333 | 999 |
| **total** | **4,489** | **2,239** | **6,728** |

The dataset is **enriched with profanity** (about one third of the examples); in real review streams profanity is far rarer, so precision on real traffic will be lower than on this test set. Almost all positives come from app reviews, and the positive class is dominated by a few frequent roots; both limitations are discussed in the README.

## Environment

In [ ]:
from importlib.metadata import version, PackageNotFoundError

for pkg in ["pandas", "numpy", "datasets", "huggingface_hub"]:
    try:
        print(f"{pkg}=={version(pkg)}")
    except PackageNotFoundError:
        pass

pandas==2.2.3
numpy==2.4.6
datasets==4.8.5
huggingface_hub==1.33.0
